In [1]:
import pandas as pd
import numpy as np

pd.options.mode.chained_assignment = None # avoids SettingWithCopy warnings

In [2]:
# returns wue (water use effectiveness) as a function of cdd (cooling degree days)
# args: cdd (vector), min_wue, max_wue
def cdd2wue(cdd, min_wue, max_wue):
    # cooling degree days (cdd) normalized to [0,1]
    cdd_norm = (cdd - min(cdd)) / (max(cdd) - min(cdd))
    
    # wue as a function of normalized cdd
    wue = min_wue + (max_wue - min_wue) * cdd_norm
    
    return wue # returns a vector

In [3]:
# returns water use in million gallons per day (mgd)
# args: power use in MW (vector), wue (vector)
def mw2mgd(power_mw, wue):
    kwh_pd = power_mw * 1000 * 24 # kwh per day
    lpd = kwh_pd * wue            # liters per day
    mgd = lpd / (3.785*1e6)       # million gallons per day
    
    return round(mgd, 4) # returns a vector

In [4]:
# read input data
file = r"D:\UC Davis\employment\Datacenter Research\Data\Analysis\california_datacenters_for_arcgis_upload.csv"
data = pd.read_csv(file)
print(data.columns)
print(f"{len(data)} California datacenters")

Index(['name', 'id', 'parent_id', 'company_name', 'company_id', 'stage',
       'type', 'type_code', 'latitude', 'longitude', 'address', 'postal',
       'city', 'county', 'market', 'state', 'country', 'whitespace_sqft',
       'whitespace_sqm', 'building_sqft', 'building_sqm', 'power_mw', 'pue',
       'year_operational', 'cooling_degree_days'],
      dtype='object')
327 California datacenters


In [5]:
# filter for hyperscale and campus DCs (Note: some colocation reclassified based on MW)
data_hsc = data[data.type_code == 3]

# impute missing MW with minimum power for a hyperscaler (=100 MW)
avg_mw = round(data_hsc["power_mw"].mean(), 2)
print(f"average power_mw = {avg_mw} MW")
data_hsc["power_mw"] = data_hsc["power_mw"].fillna(100)

# delete duplicate records (buildings within campus DC)
data_hsc = data_hsc[~data_hsc['parent_id'].isin(data_hsc['id'])]

print(f"{len(data_hsc)} unique hyperscale and campus DCs")

average power_mw = 203.17 MW
15 unique hyperscale and campus DCs


In [6]:
# WUE assumptions for hyperscalers (L/kWh)
min_wue = 1.5 # cool climate
max_wue = 3.0 # hot climate

In [7]:
# run MGD calc on all HSC DCs > convert to annual acre-feet
wue = cdd2wue(data_hsc["cooling_degree_days"], min_wue, max_wue)
data_hsc["water_mgd"] = mw2mgd(data_hsc["power_mw"], wue)
data_hsc["water_annual_af"] = data_hsc["water_mgd"] * 365 * 3.0689

# print summary
avg_water_use = round(np.mean(data_hsc["water_mgd"]), 2)
min_water_use = round(np.min(data_hsc["water_mgd"]), 2)
max_water_use = round(np.max(data_hsc["water_mgd"]), 2)
print(f"avg mgd = {avg_water_use} | min mgd = {min_water_use} | max mgd = {max_water_use}")

# export to CSV
data_hsc.head()
data_hsc.to_csv(r"D:\UC Davis\employment\Datacenter Research\Data\Analysis\hsc_datacenters_water_use.csv", index=False)

avg mgd = 2.59 | min mgd = 0.31 | max mgd = 9.51
